Models can be requested to provide their response in a format matching a given schema. This is usefulf for ensuring the outpu can easily parsed and used in subsequent processing 

In [ ]:
# Pydantic

In [1]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str =Field(description="The title of the movie")
    year: int=Field(description="This year the movie was released")
    director: str=Field(description="The director of the movie")
    rating: float=Field(description="The movie rating out of 10")



In [3]:
model_with_structure = model.with_structured_output(Movie)


In [4]:
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7fb8a5dd4980>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7fb8a5dd5400>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'descr

In [5]:
model_with_structure.invoke("Provide detail aobut the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output along parsed structure 

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str =Field(description="The title of the movie")
    year: int=Field(description="This year the movie was released")
    director: str=Field(description="The director of the movie")
    rating: float=Field(description="The movie rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

In [10]:
response = model_with_structure.invoke("Provide a detail about a random sci fi movie")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants a detail about a random sci-fi movie. Let me see. The available tool is the Movie function, which requires title, year, director, and rating. Since the user said "random," I need to pick a well-known sci-fi movie. Maybe "Blade Runner 2049"? It\'s popular and fits the genre. The director is Denis Villeneuve. It was released in 2017. The rating is around 8.0. Let me check if all parameters are covered. Title: Blade Runner 2049, Year: 2017, Director: Denis Villeneuve, Rating: 8.0. That meets all required fields. I\'ll structure the tool call with these details.\n', 'tool_calls': [{'id': 's773txcmr', 'function': {'arguments': '{"director":"Denis Villeneuve","rating":8,"title":"Blade Runner 2049","year":2017}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 215, 'prompt_tokens': 227, 'total_tokens': 442, 'completion_time': 0.363975746, 'completion_tok

# Nested Structure 

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role : str 

class MovieDetails(BaseModel):
    title: str
    year: int 
    cast : list[Actor]
    genres: list[str]
    budget: float | None =  Field(None, description="Budget in million USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details abou the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Balthazar')], genres=['Science Fiction', 'Action'], budget=160.0)

# TypedDict

In [15]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]

model_with_structure = model.with_structured_output(MovieDict)
model_with_structure.invoke("Please provide details about the moview aventgers")

{'title': 'Avengers', 'year': 2012}

In [16]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

# DataClasses 